# LESSON 4.6: Selective Filtering (Bandreject, Bandpass, and Notch Filters)
## Filtering in the Frequency Domain

In this lesson:
- Why we need selective frequency filtering
- Band-Reject Filters: Ideal, Gaussian, Butterworth
- Band-Pass Filters from Band-Reject Filters
- Notch Filters: targeting specific frequency pairs
- Application: Removing periodic noise from images
- Application: Reducing moire patterns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Introduction: Why Selective Filtering?

In previous lessons we studied **lowpass** and **highpass** filters:
- **Lowpass**: pass low frequencies (smooth/blur)
- **Highpass**: pass high frequencies (sharpen/edges)

However, many real-world problems require targeting a **specific band** or **specific frequency locations**:

| Problem | Required Filter |
|---------|----------------|
| Remove a known interference frequency | Band-Reject |
| Isolate a specific frequency band | Band-Pass |
| Remove periodic noise (bright dots in spectrum) | Notch Reject |
| Isolate periodic noise for analysis | Notch Pass |

### Biomedical Context
In biomedical imaging, periodic interference is common:
- Electrical interference (50/60 Hz) in X-ray or ultrasound images
- Moire patterns when digitizing printed medical images
- Periodic artifacts from MRI gradient coils

Selective filters allow us to **surgically remove** these artifacts without destroying the useful image content.

In [ ]:
# Utility function: create a distance matrix D(u,v) from center
def distance_from_center(shape):
    """Create a matrix of distances from the center of the frequency rectangle."""
    M, N = shape
    u = np.arange(M) - M // 2
    v = np.arange(N) - N // 2
    V, U = np.meshgrid(v, u)
    D = np.sqrt(U**2 + V**2)
    return D

# Utility function: create a test image with a circle and rectangle
def create_test_image(size=256):
    """Create a synthetic test image with geometric shapes."""
    img = np.zeros((size, size), dtype=np.float64)
    # Background gradient
    Y, X = np.ogrid[0:size, 0:size]
    img += 40
    # Circle
    cx, cy, r = size//2, size//2, size//5
    mask_circle = (X - cx)**2 + (Y - cy)**2 <= r**2
    img[mask_circle] = 200
    # Rectangle
    img[size//6:size//4, size//6:size//3] = 160
    # Small bright square
    img[3*size//4:3*size//4+20, size//4:size//4+20] = 230
    return img

# Utility function: apply frequency domain filter and show result
def apply_filter_and_show(img, H, filter_name, fig_title=None):
    """Apply a frequency domain filter H to image and display results."""
    # Compute centered DFT
    F = np.fft.fftshift(np.fft.fft2(img))
    # Apply filter
    G = F * H
    # Inverse DFT
    g = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Input Image', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(np.log1p(np.abs(F)), cmap='gray')
    axes[0, 1].set_title('Input Spectrum (log)', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, 2].set_title(f'Filter H(u,v): {filter_name}', fontsize=12)
    axes[0, 2].axis('off')
    
    axes[1, 0].imshow(g, cmap='gray')
    axes[1, 0].set_title('Filtered Image', fontsize=12)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(np.log1p(np.abs(G)), cmap='gray')
    axes[1, 1].set_title('Filtered Spectrum (log)', fontsize=12)
    axes[1, 1].axis('off')
    
    # Show filter profile through center
    center_row = H.shape[0] // 2
    axes[1, 2].plot(H[center_row, :], 'b-', linewidth=2)
    axes[1, 2].set_title('Filter Profile (center row)', fontsize=12)
    axes[1, 2].set_xlabel('Frequency')
    axes[1, 2].set_ylabel('H(u,v)')
    axes[1, 2].set_ylim([-0.05, 1.1])
    axes[1, 2].grid(True, alpha=0.3)
    
    title = fig_title if fig_title else filter_name
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return g

print("Utility functions defined.")

## 2. Band-Reject (Bandstop) Filters

A **band-reject filter** removes (attenuates) frequencies within a specific band while passing all others.

The filter is defined by two parameters:
- $D_0$: the **center radius** of the band (distance from origin in frequency domain)
- $W$: the **width** of the band

The band to be rejected lies between radii $D_0 - W/2$ and $D_0 + W/2$.

### Geometric Interpretation
In the centered frequency domain, a band-reject filter looks like a **ring-shaped dark region** (reject zone) on a white background (pass zone). All frequencies whose distance $D(u,v)$ from the center falls within the ring are suppressed.

### 2.1 Ideal Band-Reject Filter (IBRF)

$$H_{\text{IBRF}}(u,v) = \begin{cases} 0 & \text{if } D_0 - \frac{W}{2} \le D(u,v) \le D_0 + \frac{W}{2} \\ 1 & \text{otherwise} \end{cases}$$

Where $D(u,v) = \sqrt{(u - M/2)^2 + (v - N/2)^2}$ is the distance from the center of the (centered) frequency rectangle.

**Properties:**
- Sharp cutoff at band boundaries
- Introduces **ringing artifacts** due to the abrupt transition
- Not used in practice, but useful for understanding the concept

In [ ]:
def ideal_band_reject(shape, D0, W):
    """Ideal Band-Reject Filter.
    D0: center radius of the rejected band
    W: width of the rejected band
    """
    D = distance_from_center(shape)
    H = np.ones(shape, dtype=np.float64)
    band_mask = (D >= D0 - W/2) & (D <= D0 + W/2)
    H[band_mask] = 0.0
    return H

# Demonstrate the Ideal Band-Reject filter transfer function
size = 256
D0, W = 60, 30

H_ibr = ideal_band_reject((size, size), D0, W)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(H_ibr, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Ideal Band-Reject Filter\n$D_0$={D0}, W={W}', fontsize=12)
axes[0].axis('off')

# Profile through center
center = size // 2
axes[1].plot(H_ibr[center, :], 'b-', linewidth=2)
axes[1].axvline(x=center - D0, color='r', linestyle='--', alpha=0.7, label=f'$D_0$={D0}')
axes[1].axvline(x=center + D0, color='r', linestyle='--', alpha=0.7)
axes[1].axvline(x=center - D0 + W//2, color='g', linestyle=':', alpha=0.7, label=f'W={W}')
axes[1].axvline(x=center - D0 - W//2, color='g', linestyle=':', alpha=0.7)
axes[1].axvline(x=center + D0 - W//2, color='g', linestyle=':', alpha=0.7)
axes[1].axvline(x=center + D0 + W//2, color='g', linestyle=':', alpha=0.7)
axes[1].set_title('Filter Profile (center row)', fontsize=12)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('H(u,v)')
axes[1].set_ylim([-0.05, 1.1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Ideal Band-Reject Filter: Ring-Shaped Reject Zone', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Apply Ideal Band-Reject filter to a test image
test_img = create_test_image(256)
H_ibr = ideal_band_reject((256, 256), D0=60, W=30)
_ = apply_filter_and_show(test_img, H_ibr, 'Ideal Band-Reject',
                          'Ideal Band-Reject Filter ($D_0$=60, W=30)')

### 2.2 Gaussian Band-Reject Filter (GBRF)

$$H_{\text{GBRF}}(u,v) = 1 - e^{-\frac{1}{2}\left[\frac{D^2(u,v) - D_0^2}{D(u,v) \cdot W}\right]^2}$$

**Properties:**
- Smooth transition from pass to reject
- **No ringing artifacts** (Gaussian has no side lobes)
- The band edges are smooth, not abrupt

In [ ]:
def gaussian_band_reject(shape, D0, W):
    """Gaussian Band-Reject Filter.
    D0: center radius of the rejected band
    W: width of the rejected band
    """
    D = distance_from_center(shape)
    # Avoid division by zero at the center
    D_safe = np.where(D == 0, 1e-10, D)
    exponent = -0.5 * ((D**2 - D0**2) / (D_safe * W))**2
    H = 1.0 - np.exp(exponent)
    return H

# Demonstrate the Gaussian Band-Reject filter
H_gbr = gaussian_band_reject((256, 256), D0=60, W=30)
_ = apply_filter_and_show(test_img, H_gbr, 'Gaussian Band-Reject',
                          'Gaussian Band-Reject Filter ($D_0$=60, W=30)')

### 2.3 Butterworth Band-Reject Filter (BBRF)

$$H_{\text{BBRF}}(u,v) = \frac{1}{1 + \left[\frac{D(u,v) \cdot W}{D^2(u,v) - D_0^2}\right]^{2n}}$$

Where $n$ is the **order** of the filter.

**Properties:**
- Smooth transition, controlled by the order $n$
- As $n \to \infty$, approaches the ideal band-reject filter
- For low $n$, the transition is very gradual
- Good compromise between ringing and selectivity

In [ ]:
def butterworth_band_reject(shape, D0, W, n=2):
    """Butterworth Band-Reject Filter.
    D0: center radius of the rejected band
    W: width of the rejected band
    n: order of the filter
    """
    D = distance_from_center(shape)
    # Avoid division by zero where D^2 == D0^2
    denom = D**2 - D0**2
    denom = np.where(denom == 0, 1e-10, denom)
    H = 1.0 / (1.0 + ((D * W) / denom) ** (2 * n))
    return H

# Demonstrate the Butterworth Band-Reject filter
H_bbr = butterworth_band_reject((256, 256), D0=60, W=30, n=2)
_ = apply_filter_and_show(test_img, H_bbr, 'Butterworth Band-Reject (n=2)',
                          'Butterworth Band-Reject Filter ($D_0$=60, W=30, n=2)')

### 2.4 Comparison of Band-Reject Filters

Let us compare the three band-reject filter types side by side, examining both their transfer functions and their cross-sectional profiles.

In [ ]:
# Compare all three band-reject filters
size = 256
D0, W = 60, 30

H_ideal = ideal_band_reject((size, size), D0, W)
H_gauss = gaussian_band_reject((size, size), D0, W)
H_butter = butterworth_band_reject((size, size), D0, W, n=2)

center = size // 2

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

filters = [H_ideal, H_gauss, H_butter]
names = ['Ideal', 'Gaussian', 'Butterworth (n=2)']

for i, (H, name) in enumerate(zip(filters, names)):
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'{name} Band-Reject', fontsize=12)
    axes[0, i].axis('off')

# Overlay profiles
for i, (H, name) in enumerate(zip(filters, names)):
    axes[1, i].plot(H[center, :], 'b-', linewidth=2)
    axes[1, i].set_title(f'{name} Profile', fontsize=12)
    axes[1, i].set_xlabel('Frequency')
    axes[1, i].set_ylabel('H(u,v)')
    axes[1, i].set_ylim([-0.05, 1.1])
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle(f'Comparison of Band-Reject Filters ($D_0$={D0}, W={W})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Effect of varying bandwidth W
size = 256
D0 = 60
W_values = [10, 30, 60]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, W in enumerate(W_values):
    H = butterworth_band_reject((size, size), D0, W, n=2)
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'W = {W}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].plot(H[size//2, :], 'b-', linewidth=2)
    axes[1, i].set_title(f'Profile (W={W})', fontsize=12)
    axes[1, i].set_xlabel('Frequency')
    axes[1, i].set_ylabel('H(u,v)')
    axes[1, i].set_ylim([-0.05, 1.1])
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle(f'Effect of Bandwidth W on Butterworth Band-Reject ($D_0$={D0}, n=2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Wider W -> more frequencies rejected -> more image information lost")

## 3. Band-Pass Filters

A **band-pass filter** is the complement of a band-reject filter. It passes **only** the frequencies within the band and rejects everything else:

$$\boxed{H_{\text{BP}}(u,v) = 1 - H_{\text{BR}}(u,v)}$$

This simple relationship means:
- **Ideal Band-Pass** = $1 -$ Ideal Band-Reject
- **Gaussian Band-Pass** = $1 -$ Gaussian Band-Reject
- **Butterworth Band-Pass** = $1 -$ Butterworth Band-Reject

### Use cases:
- Isolating a specific frequency component for analysis
- Extracting periodic patterns from images
- Visualizing what information exists in a particular frequency band

**Important:** Band-pass filters remove the DC component ($F(0,0)$), so the filtered image will have zero average intensity. The result shows only the **variations** in the selected band.

In [ ]:
def ideal_band_pass(shape, D0, W):
    """Ideal Band-Pass Filter = 1 - Ideal Band-Reject."""
    return 1.0 - ideal_band_reject(shape, D0, W)

def gaussian_band_pass(shape, D0, W):
    """Gaussian Band-Pass Filter = 1 - Gaussian Band-Reject."""
    return 1.0 - gaussian_band_reject(shape, D0, W)

def butterworth_band_pass(shape, D0, W, n=2):
    """Butterworth Band-Pass Filter = 1 - Butterworth Band-Reject."""
    return 1.0 - butterworth_band_reject(shape, D0, W, n)

# Compare Band-Reject and Band-Pass
size = 256
D0, W = 60, 30

H_br = butterworth_band_reject((size, size), D0, W, n=2)
H_bp = butterworth_band_pass((size, size), D0, W, n=2)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(H_br, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Band-Reject Filter', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(H_bp, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Band-Pass Filter (= 1 - BR)', fontsize=12)
axes[0, 1].axis('off')

center = size // 2
axes[1, 0].plot(H_br[center, :], 'b-', linewidth=2, label='Band-Reject')
axes[1, 0].set_title('Band-Reject Profile', fontsize=12)
axes[1, 0].set_xlabel('Frequency')
axes[1, 0].set_ylabel('H(u,v)')
axes[1, 0].set_ylim([-0.05, 1.1])
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(H_bp[center, :], 'r-', linewidth=2, label='Band-Pass')
axes[1, 1].set_title('Band-Pass Profile', fontsize=12)
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].set_ylabel('H(u,v)')
axes[1, 1].set_ylim([-0.05, 1.1])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Band-Reject vs Band-Pass: $H_{BP} = 1 - H_{BR}$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Apply Band-Pass filter to test image
# This isolates the frequency content in the band
test_img = create_test_image(256)
H_bp = butterworth_band_pass((256, 256), D0=40, W=20, n=2)
_ = apply_filter_and_show(test_img, H_bp, 'Butterworth Band-Pass',
                          'Band-Pass Isolates a Specific Frequency Band ($D_0$=40, W=20)')

In [ ]:
# Decompose image into multiple frequency bands
test_img = create_test_image(256)
shape = test_img.shape
F = np.fft.fftshift(np.fft.fft2(test_img))

# Define frequency bands
bands = [
    {'D0': 10, 'W': 20, 'label': 'Very Low (0-20)'},
    {'D0': 30, 'W': 20, 'label': 'Low (20-40)'},
    {'D0': 55, 'W': 30, 'label': 'Medium (40-70)'},
    {'D0': 90, 'W': 40, 'label': 'High (70-110)'},
]

fig, axes = plt.subplots(2, len(bands), figsize=(16, 8))

for i, band in enumerate(bands):
    H_bp = butterworth_band_pass(shape, band['D0'], band['W'], n=2)
    G = F * H_bp
    g = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    
    axes[0, i].imshow(H_bp, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f"Band: {band['label']}", fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(g, cmap='gray')
    axes[1, i].set_title('Band Content', fontsize=11)
    axes[1, i].axis('off')

plt.suptitle('Image Decomposition into Frequency Bands\n'
             'Low bands: smooth regions | High bands: edges and details',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Notch Filters

While band-reject filters remove an **annular ring** of frequencies (all frequencies at a given distance from center), **notch filters** target specific **point locations** in the frequency domain.

### Why notch filters?
Periodic noise in an image appears as **bright dots (spikes)** at specific locations in the frequency spectrum. A notch filter can suppress these spikes without affecting the rest of the spectrum.

### Key principle: Conjugate symmetry
Because images are **real-valued**, the DFT has conjugate symmetry:

$$F^*(u,v) = F(-u,-v)$$

This means noise spikes always appear in **symmetric pairs**. A notch filter must therefore always suppress frequencies at both $(u_0, v_0)$ and $(-u_0, -v_0)$ to preserve the real-valued nature of the result.

### Notch Reject Filter

For a notch centered at $(u_0, v_0)$ and its conjugate $(-u_0, -v_0)$:

$$H_{NR}(u,v) = \prod_{k=1}^{Q} H_k(u,v) \cdot H_{-k}(u,v)$$

where each pair creates a notch at one spike and its conjugate.

### Notch Pass Filter

$$H_{NP}(u,v) = 1 - H_{NR}(u,v)$$

### 4.1 Notch Filter Construction

For a single notch pair at $(u_0, v_0)$ and $(-u_0, -v_0)$ in the centered frequency domain:

#### Distance from each notch center:
$$D_1(u,v) = \sqrt{(u - M/2 - u_0)^2 + (v - N/2 - v_0)^2}$$
$$D_2(u,v) = \sqrt{(u - M/2 + u_0)^2 + (v - N/2 + v_0)^2}$$

#### Ideal Notch Reject:
$$H_{NR}(u,v) = \begin{cases} 0 & \text{if } D_1(u,v) \le D_0 \text{ or } D_2(u,v) \le D_0 \\ 1 & \text{otherwise} \end{cases}$$

#### Butterworth Notch Reject:
$$H_{NR}(u,v) = \frac{1}{1 + \left[\frac{D_0^2}{D_1(u,v) \cdot D_2(u,v)}\right]^n}$$

#### Gaussian Notch Reject:
$$H_{NR}(u,v) = 1 - e^{-\frac{1}{2}\frac{D_1(u,v) \cdot D_2(u,v)}{D_0^2}}$$

In [ ]:
def notch_reject_ideal(shape, notch_centers, D0):
    """Ideal Notch Reject Filter.
    shape: (M, N) image dimensions
    notch_centers: list of (u0, v0) offsets from center
    D0: radius of the notch
    """
    M, N = shape
    u = np.arange(M)
    v = np.arange(N)
    V, U = np.meshgrid(v, u)
    
    H = np.ones(shape, dtype=np.float64)
    
    for u0, v0 in notch_centers:
        # Distance from (M/2 + u0, N/2 + v0)
        D1 = np.sqrt((U - M//2 - u0)**2 + (V - N//2 - v0)**2)
        # Distance from conjugate (M/2 - u0, N/2 - v0)
        D2 = np.sqrt((U - M//2 + u0)**2 + (V - N//2 + v0)**2)
        H[(D1 <= D0) | (D2 <= D0)] = 0.0
    
    return H

def notch_reject_butterworth(shape, notch_centers, D0, n=2):
    """Butterworth Notch Reject Filter.
    shape: (M, N) image dimensions
    notch_centers: list of (u0, v0) offsets from center
    D0: radius of the notch
    n: order of the filter
    """
    M, N = shape
    u = np.arange(M)
    v = np.arange(N)
    V, U = np.meshgrid(v, u)
    
    H = np.ones(shape, dtype=np.float64)
    
    for u0, v0 in notch_centers:
        D1 = np.sqrt((U - M//2 - u0)**2 + (V - N//2 - v0)**2)
        D2 = np.sqrt((U - M//2 + u0)**2 + (V - N//2 + v0)**2)
        # Avoid division by zero
        D1D2 = D1 * D2
        D1D2 = np.where(D1D2 == 0, 1e-10, D1D2)
        H_k = 1.0 / (1.0 + (D0**2 / D1D2) ** n)
        H *= H_k
    
    return H

def notch_reject_gaussian(shape, notch_centers, D0):
    """Gaussian Notch Reject Filter.
    shape: (M, N) image dimensions
    notch_centers: list of (u0, v0) offsets from center
    D0: radius of the notch
    """
    M, N = shape
    u = np.arange(M)
    v = np.arange(N)
    V, U = np.meshgrid(v, u)
    
    H = np.ones(shape, dtype=np.float64)
    
    for u0, v0 in notch_centers:
        D1 = np.sqrt((U - M//2 - u0)**2 + (V - N//2 - v0)**2)
        D2 = np.sqrt((U - M//2 + u0)**2 + (V - N//2 + v0)**2)
        D1D2 = D1 * D2
        H_k = 1.0 - np.exp(-0.5 * D1D2 / D0**2)
        H *= H_k
    
    return H

print("Notch filter functions defined.")

In [ ]:
# Visualize notch filters with a single notch pair
size = 256
notch_centers = [(30, 40)]  # One pair: (30,40) and (-30,-40)
D0 = 15

H_ideal = notch_reject_ideal((size, size), notch_centers, D0)
H_butter = notch_reject_butterworth((size, size), notch_centers, D0, n=2)
H_gauss = notch_reject_gaussian((size, size), notch_centers, D0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, H, name in zip(axes, [H_ideal, H_butter, H_gauss],
                        ['Ideal', 'Butterworth (n=2)', 'Gaussian']):
    ax.imshow(H, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{name} Notch Reject\nNotch at (30,40) and (-30,-40)', fontsize=11)
    ax.axis('off')

plt.suptitle(f'Notch Reject Filters ($D_0$={D0}): Symmetric Pair of Notches',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Notch Reject vs Notch Pass
size = 256
notch_centers = [(30, 40), (0, 50)]  # Two pairs of notches
D0 = 12

H_nr = notch_reject_butterworth((size, size), notch_centers, D0, n=2)
H_np = 1.0 - H_nr  # Notch pass

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(H_nr, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Notch Reject $H_{NR}(u,v)$\n(suppress specific frequencies)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(H_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Notch Pass $H_{NP} = 1 - H_{NR}$\n(isolate specific frequencies)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Notch Reject vs Notch Pass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Application: Removing Periodic Noise from an Image

**Periodic noise** is one of the most common artifacts in biomedical images. It appears as a repetitive pattern overlaid on the image.

### The key insight:
Periodic noise in the spatial domain produces **bright dots (spikes)** at specific locations in the frequency spectrum. These locations correspond to the frequency and direction of the periodic pattern.

### Workflow:
1. Compute the DFT of the noisy image
2. Examine the spectrum to locate bright spikes (noise frequencies)
3. Design a notch filter to suppress those specific frequencies
4. Apply the filter and compute the inverse DFT

Let us create a synthetic example by adding sinusoidal noise to a clean image.

In [ ]:
# Create a clean test image
size = 256
clean_img = create_test_image(size)

# Add periodic (sinusoidal) noise
x = np.arange(size)
y = np.arange(size)
Y, X = np.meshgrid(y, x)

# Sinusoidal noise: two frequency components
# Component 1: frequency in vertical direction
freq1_u, freq1_v = 0, 40  # vertical stripes
noise1 = 40 * np.sin(2 * np.pi * freq1_v * Y / size)

# Component 2: frequency in diagonal direction
freq2_u, freq2_v = 30, 30  # diagonal stripes
noise2 = 30 * np.sin(2 * np.pi * (freq2_u * X + freq2_v * Y) / size)

# Noisy image
noisy_img = clean_img + noise1 + noise2

# Compute spectrum of noisy image
F_clean = np.fft.fftshift(np.fft.fft2(clean_img))
F_noisy = np.fft.fftshift(np.fft.fft2(noisy_img))

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(clean_img, cmap='gray')
axes[0, 0].set_title('Clean Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_img, cmap='gray')
axes[0, 1].set_title('Image + Periodic Noise', fontsize=12)
axes[0, 1].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(F_clean)), cmap='gray')
axes[1, 0].set_title('Clean Image Spectrum', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(F_noisy)), cmap='gray')
axes[1, 1].set_title('Noisy Image Spectrum\n(Note the bright dots = noise frequencies!)', fontsize=12)
axes[1, 1].axis('off')

plt.suptitle('Periodic Noise Creates Bright Spikes in the Frequency Spectrum',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The bright dots in the noisy spectrum correspond to the noise frequencies.")
print(f"Noise component 1: frequency offset (u0, v0) = ({freq1_u}, {freq1_v})")
print(f"Noise component 2: frequency offset (u0, v0) = ({freq2_u}, {freq2_v})")

In [ ]:
# Design notch filter to remove the periodic noise
# The noise spikes are at the known frequency offsets
notch_centers = [
    (freq1_u, freq1_v),   # Noise component 1: (0, 40)
    (freq2_u, freq2_v),   # Noise component 2: (30, 30)
]
D0_notch = 10  # radius of each notch

# Use Butterworth notch reject for smooth suppression
H_notch = notch_reject_butterworth((size, size), notch_centers, D0_notch, n=4)

# Apply notch filter
G = F_noisy * H_notch
filtered_img = np.real(np.fft.ifft2(np.fft.ifftshift(G)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(noisy_img, cmap='gray')
axes[0, 0].set_title('Noisy Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(np.log1p(np.abs(F_noisy)), cmap='gray')
axes[0, 1].set_title('Noisy Spectrum\n(with bright noise spikes)', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(H_notch, cmap='gray', vmin=0, vmax=1)
axes[0, 2].set_title(f'Notch Reject Filter ($D_0$={D0_notch})', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(filtered_img, cmap='gray')
axes[1, 0].set_title('Filtered Image (noise removed!)', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(G)), cmap='gray')
axes[1, 1].set_title('Filtered Spectrum\n(spikes suppressed)', fontsize=12)
axes[1, 1].axis('off')

# Show the extracted noise pattern using notch pass
H_np = 1.0 - H_notch
G_noise = F_noisy * H_np
noise_extracted = np.real(np.fft.ifft2(np.fft.ifftshift(G_noise)))
axes[1, 2].imshow(noise_extracted, cmap='gray')
axes[1, 2].set_title('Extracted Noise Pattern\n(notch pass output)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Removing Periodic Noise with Notch Filters',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: clean vs noisy vs filtered
mse_noisy = np.mean((clean_img - noisy_img)**2)
mse_filtered = np.mean((clean_img - filtered_img)**2)
improvement = (1 - mse_filtered / mse_noisy) * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(clean_img, cmap='gray')
axes[0].set_title('Clean (Ground Truth)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(noisy_img, cmap='gray')
axes[1].set_title(f'Noisy (MSE = {mse_noisy:.1f})', fontsize=12)
axes[1].axis('off')

axes[2].imshow(filtered_img, cmap='gray')
axes[2].set_title(f'Filtered (MSE = {mse_filtered:.1f})', fontsize=12)
axes[2].axis('off')

plt.suptitle(f'Notch Filter Reduces Error by {improvement:.1f}%',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"MSE (noisy vs clean):    {mse_noisy:.2f}")
print(f"MSE (filtered vs clean): {mse_filtered:.2f}")
print(f"Error reduction:         {improvement:.1f}%")

### 5.1 Effect of Notch Radius $D_0$

The notch radius $D_0$ controls how much of the spectrum is suppressed around each noise spike. A larger $D_0$ removes more noise but may also remove some useful image content.

In [ ]:
# Effect of notch radius D0
D0_values = [5, 10, 20, 40]

fig, axes = plt.subplots(2, len(D0_values), figsize=(16, 8))

for i, D0_val in enumerate(D0_values):
    H = notch_reject_butterworth((size, size), notch_centers, D0_val, n=4)
    G = F_noisy * H
    result = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    mse = np.mean((clean_img - result)**2)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'$D_0$ = {D0_val}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(result, cmap='gray')
    axes[1, i].set_title(f'MSE = {mse:.1f}', fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Effect of Notch Radius $D_0$: Larger Radius = More Suppression',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Too small D0: noise remains. Too large D0: image content is also removed.")
print("The optimal D0 balances noise removal and information preservation.")

## 6. Application: Removing Multiple Periodic Noise Sources

In practice, images may be corrupted by **multiple periodic noise sources** simultaneously. Each source creates its own pair of spikes in the spectrum. We can place multiple notches to handle all of them.

In [ ]:
# Create image with multiple periodic noise sources
size = 256
clean = create_test_image(size)

x = np.arange(size)
y = np.arange(size)
Y, X = np.meshgrid(y, x)

# Four different periodic noise components
noise_sources = [
    {'u': 0,  'v': 30,  'amp': 35, 'label': 'Vertical stripes'},
    {'u': 40, 'v': 0,   'amp': 30, 'label': 'Horizontal stripes'},
    {'u': 25, 'v': 25,  'amp': 25, 'label': 'Diagonal 1'},
    {'u': -20,'v': 35,  'amp': 20, 'label': 'Diagonal 2'},
]

noise_total = np.zeros((size, size))
notch_list = []
for src in noise_sources:
    noise_total += src['amp'] * np.sin(2 * np.pi * (src['u'] * X + src['v'] * Y) / size)
    notch_list.append((src['u'], src['v']))

noisy_multi = clean + noise_total

# Compute DFT
F_noisy_multi = np.fft.fftshift(np.fft.fft2(noisy_multi))

# Create multi-notch filter
H_multi = notch_reject_butterworth((size, size), notch_list, D0=8, n=4)

# Apply filter
G_multi = F_noisy_multi * H_multi
filtered_multi = np.real(np.fft.ifft2(np.fft.ifftshift(G_multi)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(clean, cmap='gray')
axes[0, 0].set_title('Clean Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_multi, cmap='gray')
axes[0, 1].set_title('Corrupted by 4 Noise Sources', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(np.log1p(np.abs(F_noisy_multi)), cmap='gray')
axes[0, 2].set_title('Noisy Spectrum\n(4 pairs of bright dots)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(H_multi, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Multi-Notch Filter\n(8 notch points)', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(filtered_multi, cmap='gray')
axes[1, 1].set_title('Filtered Result', fontsize=12)
axes[1, 1].axis('off')

# Error map
error = np.abs(clean - filtered_multi)
axes[1, 2].imshow(error, cmap='hot')
axes[1, 2].set_title(f'Error Map (MSE = {np.mean(error**2):.1f})', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Removing Multiple Periodic Noise Sources with Multi-Notch Filter',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Application: Reducing Moire Patterns

### What are Moire Patterns?

**Moire patterns** are visual artifacts that appear when a spatially periodic structure (such as a halftone-printed image) is sampled by another periodic process (such as a digital scanner). The interaction of the two periodic grids creates a new, undesired low-frequency pattern.

### Common sources in biomedical imaging:
- **Digitizing printed medical images** (textbooks, old patient records)
- **Scanning halftone-printed radiographs** from publications
- **Interference between sensor grid and periodic structures** in the scene

### Why frequency domain filtering helps:
The moire pattern is periodic, so it produces **distinct spikes** in the frequency spectrum. By identifying and removing these spikes with **notch filters** or **band-reject filters**, we can significantly reduce the moire artifact while preserving the underlying image.

### Typical approach:
1. Compute the DFT and examine the log spectrum
2. Identify the periodic spikes caused by the moire pattern
3. Place notch filters at those spike locations
4. Alternatively, use a band-reject filter if the moire frequencies fall in a known band
5. Apply the filter and reconstruct the image

In [ ]:
# Simulate a moire pattern scenario
# Step 1: Create a "printed" image (original with halftone-like dot pattern)
size = 256
np.random.seed(42)

# Create a smooth base image (simulating a medical image)
Y, X = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
base_img = 120 + 80 * np.exp(-(X**2 + Y**2) / 0.3)
# Add some structure
base_img += 40 * (np.sqrt(X**2 + Y**2) < 0.2).astype(float)
base_img += -30 * ((np.abs(X - 0.3) < 0.1) & (np.abs(Y + 0.3) < 0.15)).astype(float)

# Simulate halftone/printing grid pattern (high frequency periodic structure)
grid_freq = 32  # frequency of the printing grid
xx = np.arange(size)
yy = np.arange(size)
YY, XX = np.meshgrid(yy, xx)

# Two perpendicular grid patterns (simulating halftone dots)
halftone_h = 15 * np.sin(2 * np.pi * grid_freq * XX / size)
halftone_v = 15 * np.sin(2 * np.pi * grid_freq * YY / size)
halftone = halftone_h + halftone_v

# Simulated "scanned printed image" = base + halftone pattern
moire_img = base_img + halftone

# Compute spectra
F_base = np.fft.fftshift(np.fft.fft2(base_img))
F_moire = np.fft.fftshift(np.fft.fft2(moire_img))

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(base_img, cmap='gray')
axes[0, 0].set_title('Original Image\n(clean medical image)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(moire_img, cmap='gray')
axes[0, 1].set_title('Scanned Print with Moire Pattern\n(halftone interference)', fontsize=12)
axes[0, 1].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(F_base)), cmap='gray')
axes[1, 0].set_title('Clean Spectrum', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(F_moire)), cmap='gray')
axes[1, 1].set_title('Moire Spectrum\n(halftone spikes visible)', fontsize=12)
axes[1, 1].axis('off')

plt.suptitle('Moire Pattern: Periodic Halftone Creates Spectral Spikes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Remove moire pattern using notch filters
# The halftone pattern creates spikes at the grid frequencies
# Horizontal grid: spike at (grid_freq, 0) and (-grid_freq, 0)
# Vertical grid: spike at (0, grid_freq) and (0, -grid_freq)
moire_notch_centers = [
    (grid_freq, 0),   # Horizontal halftone component
    (0, grid_freq),   # Vertical halftone component
]

H_moire_notch = notch_reject_butterworth((size, size), moire_notch_centers, D0=8, n=4)

# Apply filter
G_demoire = F_moire * H_moire_notch
demoire_img = np.real(np.fft.ifft2(np.fft.ifftshift(G_demoire)))

# Also try band-reject approach
# The halftone frequency is at distance ~grid_freq from center
H_br_moire = butterworth_band_reject((size, size), D0=grid_freq, W=10, n=4)
G_br = F_moire * H_br_moire
br_result = np.real(np.fft.ifft2(np.fft.ifftshift(G_br)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(moire_img, cmap='gray')
axes[0, 0].set_title('Moire-Corrupted Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(H_moire_notch, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Notch Reject Filter', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(demoire_img, cmap='gray')
axes[0, 2].set_title(f'Notch Filtered\nMSE = {np.mean((base_img - demoire_img)**2):.1f}', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(F_moire)), cmap='gray')
axes[1, 0].set_title('Moire Spectrum', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(H_br_moire, cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title('Band-Reject Filter', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(br_result, cmap='gray')
axes[1, 2].set_title(f'Band-Reject Filtered\nMSE = {np.mean((base_img - br_result)**2):.1f}', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Moire Pattern Removal: Notch Filter vs Band-Reject Filter',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notch filter is more selective: it only removes the specific moire spikes.")
print("Band-reject filter removes an entire ring, which may discard useful content.")

In [ ]:
# Detailed view: zoom into a region to see moire removal
r1, r2, c1, c2 = 80, 180, 80, 180  # zoom region

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(base_img[r1:r2, c1:c2], cmap='gray')
axes[0].set_title('Clean (zoomed)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(moire_img[r1:r2, c1:c2], cmap='gray')
axes[1].set_title('Moire (zoomed)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(demoire_img[r1:r2, c1:c2], cmap='gray')
axes[2].set_title('Notch Filtered (zoomed)', fontsize=11)
axes[2].axis('off')

axes[3].imshow(br_result[r1:r2, c1:c2], cmap='gray')
axes[3].set_title('Band-Reject Filtered (zoomed)', fontsize=11)
axes[3].axis('off')

plt.suptitle('Zoomed Comparison: Moire Pattern Reduction',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Moire Patterns: Key Takeaways

1. **Moire patterns** arise when two periodic structures interfere (e.g., halftone dots + scanner sampling grid)
2. They are **periodic**, so they produce **predictable spikes** in the frequency spectrum
3. **Notch filters** provide the most selective removal: they target only the moire spikes
4. **Band-reject filters** can also work but remove a broader range of frequencies
5. In practice, identifying the correct spike locations may require **visual inspection** of the spectrum

## 8. Comparison: Band-Reject vs Notch Filters

| Feature | Band-Reject | Notch |
|---------|------------|-------|
| **Shape in frequency domain** | Ring (annulus) | Small circles at specific points |
| **Frequencies affected** | All at distance $D_0$ from center | Only at specific $(u_0, v_0)$ pairs |
| **Selectivity** | Low (removes entire band) | High (targets specific spikes) |
| **Best for** | Unknown noise spread in a band | Known periodic interference |
| **Preserves image content** | Less (removes broad band) | More (removes only spikes) |
| **Conjugate symmetry** | Automatic (ring is symmetric) | Must place pairs at $(u_0,v_0)$ and $(-u_0,-v_0)$ |

In [ ]:
# Visual comparison: Band-Reject vs Notch for the same noise
size = 256
clean = create_test_image(size)

xx = np.arange(size)
yy = np.arange(size)
YY, XX = np.meshgrid(yy, xx)

# Add periodic noise at a specific frequency
noise_u, noise_v = 40, 30
noise = 40 * np.sin(2 * np.pi * (noise_u * XX + noise_v * YY) / size)
noisy = clean + noise

F = np.fft.fftshift(np.fft.fft2(noisy))

# The noise distance from center
D_noise = np.sqrt(noise_u**2 + noise_v**2)

# Band-reject centered at that distance
H_br = butterworth_band_reject((size, size), D0=D_noise, W=15, n=4)
result_br = np.real(np.fft.ifft2(np.fft.ifftshift(F * H_br)))

# Notch at the exact spike location
H_notch = notch_reject_butterworth((size, size), [(noise_u, noise_v)], D0=10, n=4)
result_notch = np.real(np.fft.ifft2(np.fft.ifftshift(F * H_notch)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(noisy, cmap='gray')
axes[0, 0].set_title('Noisy Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(H_br, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title(f'Band-Reject (ring at D={D_noise:.0f})', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(H_notch, cmap='gray', vmin=0, vmax=1)
axes[0, 2].set_title(f'Notch (at u={noise_u}, v={noise_v})', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(F)), cmap='gray')
axes[1, 0].set_title('Noisy Spectrum', fontsize=12)
axes[1, 0].axis('off')

mse_br = np.mean((clean - result_br)**2)
axes[1, 1].imshow(result_br, cmap='gray')
axes[1, 1].set_title(f'Band-Reject Result\nMSE = {mse_br:.1f}', fontsize=12)
axes[1, 1].axis('off')

mse_notch = np.mean((clean - result_notch)**2)
axes[1, 2].imshow(result_notch, cmap='gray')
axes[1, 2].set_title(f'Notch Result\nMSE = {mse_notch:.1f}', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Band-Reject (removes ring) vs Notch (removes points): Notch is More Selective',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Band-Reject MSE: {mse_br:.2f}")
print(f"Notch MSE:       {mse_notch:.2f}")
print(f"Notch preserves {((mse_br - mse_notch)/mse_br)*100:.1f}% more image content.")

## 9. Practical Workflow: Identifying Noise Frequencies

In real applications, we may not know the exact noise frequencies beforehand. The practical workflow is:

1. **Compute and display the log spectrum** of the noisy image
2. **Visually identify** the bright dots that do not belong to the image content
3. **Determine their coordinates** relative to the center of the spectrum
4. **Design the notch filter** based on those coordinates
5. **Iterate**: adjust $D_0$ and check results

Below we demonstrate an automated spike detection approach using thresholding on the spectrum.

In [ ]:
# Automated spike detection and notch filtering
size = 256
clean = create_test_image(size)

xx = np.arange(size)
yy = np.arange(size)
YY, XX = np.meshgrid(yy, xx)

# Add several periodic noise components
noise = (35 * np.sin(2 * np.pi * (20 * XX + 50 * YY) / size) +
         30 * np.sin(2 * np.pi * (45 * XX - 15 * YY) / size) +
         25 * np.sin(2 * np.pi * (0 * XX + 60 * YY) / size))
noisy = clean + noise

# Compute spectrum
F = np.fft.fftshift(np.fft.fft2(noisy))
mag = np.abs(F)
log_mag = np.log1p(mag)

# Simple spike detection: find bright points far from center
# Exclude the DC region
D = distance_from_center((size, size))
mag_masked = mag.copy()
mag_masked[D < 5] = 0  # ignore DC neighborhood

# Threshold: find points that are much brighter than their neighborhood
threshold = np.percentile(mag_masked[mag_masked > 0], 99.5)
spike_mask = mag_masked > threshold

# Find spike locations
spike_coords = np.argwhere(spike_mask)
center = size // 2

# Convert to offsets from center and keep only unique pairs (positive u0)
detected_notches = []
for coord in spike_coords:
    u0 = coord[0] - center
    v0 = coord[1] - center
    if u0 > 0 or (u0 == 0 and v0 > 0):  # keep one from each conjugate pair
        detected_notches.append((u0, v0))

print(f"Detected {len(detected_notches)} noise spike pairs:")
for u0, v0 in detected_notches:
    print(f"  ({u0}, {v0}) and ({-u0}, {-v0})")

# Build notch filter from detected spikes
H_auto = notch_reject_butterworth((size, size), detected_notches, D0=8, n=4)
result_auto = np.real(np.fft.ifft2(np.fft.ifftshift(F * H_auto)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(noisy, cmap='gray')
axes[0, 0].set_title('Noisy Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(log_mag, cmap='gray')
axes[0, 1].set_title('Log Spectrum', fontsize=12)
axes[0, 1].axis('off')

# Mark detected spikes on spectrum
axes[0, 2].imshow(log_mag, cmap='gray')
for coord in spike_coords:
    circle = plt.Circle((coord[1], coord[0]), 8, color='red',
                        fill=False, linewidth=2)
    axes[0, 2].add_patch(circle)
axes[0, 2].set_title('Detected Spikes (red circles)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(H_auto, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Auto-Designed Notch Filter', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(result_auto, cmap='gray')
axes[1, 1].set_title(f'Filtered Result\nMSE = {np.mean((clean - result_auto)**2):.1f}', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(clean, cmap='gray')
axes[1, 2].set_title('Ground Truth', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Automated Spike Detection and Notch Filtering',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

What we learned in this lesson:

1. **Band-Reject Filters** suppress a ring-shaped band of frequencies:
   - Ideal: sharp cutoff, causes ringing
   - Gaussian: smooth transition, no ringing
   - Butterworth: adjustable transition via order $n$

2. **Band-Pass Filters** are the complement: $H_{BP} = 1 - H_{BR}$
   - Useful for isolating specific frequency bands
   - Remove DC, so result shows only band content

3. **Notch Filters** target specific frequency pairs $(u_0, v_0)$ and $(-u_0, -v_0)$:
   - Much more selective than band-reject filters
   - Ideal for removing periodic noise (bright spikes in spectrum)
   - Must maintain conjugate symmetry for real-valued output

4. **Periodic noise removal** workflow:
   - Compute DFT and identify noise spikes in the spectrum
   - Design notch filter at those locations
   - Apply filter and reconstruct

5. **Moire pattern reduction**:
   - Moire arises from interference of periodic structures
   - Creates predictable spikes in the frequency domain
   - Notch filters provide selective removal

### Key formulas:

| Filter | Transfer Function |
|--------|------------------|
| Ideal BR | $H = 0$ if $D_0 - W/2 \le D \le D_0 + W/2$, else $H = 1$ |
| Gaussian BR | $H = 1 - \exp\left[-\frac{1}{2}\left(\frac{D^2 - D_0^2}{DW}\right)^2\right]$ |
| Butterworth BR | $H = \frac{1}{1 + \left(\frac{DW}{D^2 - D_0^2}\right)^{2n}}$ |
| Band-Pass | $H_{BP} = 1 - H_{BR}$ |
| Notch Reject | Product of notch pairs at each spike location |
| Notch Pass | $H_{NP} = 1 - H_{NR}$ |